# 02 — Exploratory Data Analysis

Self-contained notebook: no `%run` dependency on project scripts.

In [ ]:
from pathlib import Path
import json
import warnings
import numpy as np
import pandas as pd
import yaml

# In Colab, change only this path after mounting Google Drive.
PROJECT_ROOT = Path("..").resolve() if Path.cwd().name == "src" else Path(".").resolve()
CONFIG_PATH = PROJECT_ROOT / "config" / "config.yaml"
with CONFIG_PATH.open(encoding="utf-8") as handle:
    CONFIG = yaml.safe_load(handle)

def project_path(key):
    return PROJECT_ROOT / CONFIG["paths"][key]

OUTPUT = project_path("output")
PROCESSED = project_path("processed")
for folder in [OUTPUT / "reports", OUTPUT / "figures", OUTPUT / "models", OUTPUT / "labeling", OUTPUT / "logs", PROCESSED]:
    folder.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)

In [ ]:
import os
os.environ.setdefault("MPLCONFIGDIR", str(OUTPUT / ".matplotlib"))
import matplotlib.pyplot as plt

shops = pd.read_csv(PROCESSED / "shops_clean.csv")
products = pd.read_csv(PROCESSED / "products_clean.csv")
reviews = pd.read_csv(PROCESSED / "reviews_clean.csv")

def plot_distribution(frame, column, filename, categorical=False):
    if frame.empty:
        print(f"SKIP {column}: no data")
        return
    ax = (frame[column].value_counts().sort_index().plot.bar(title=column)
          if categorical else frame[column].plot.hist(bins=min(20, max(1, len(frame))), title=column))
    ax.set_xlabel(column); ax.figure.tight_layout()
    ax.figure.savefig(OUTPUT / "figures" / filename, dpi=150)
    plt.show(); plt.close(ax.figure)

In [ ]:
for column, name, categorical in [
    ("Shop_type", "shop_type.png", True), ("Years_Active", "years_active.png", False),
    ("Follower_Count_k", "followers.png", False), ("Rating_Average", "shop_rating.png", False),
    ("Chat_Response_Rate_pct", "response_rate.png", False)]:
    plot_distribution(shops, column, name, categorical)

plot_distribution(products, "Review_stars", "product_rating.png")
plot_distribution(reviews, "rating", "review_rating.png")
plot_distribution(reviews, "Comment_Length", "comment_length.png")
plot_distribution(reviews, "has_image", "has_image.png", True)

summary = pd.DataFrame({
    "Metric": ["Total Shops", "Total Products", "Total Reviews", "Average Review Rating", "Image Rate"],
    "Value": [len(shops), len(products), len(reviews), reviews["rating"].mean(), reviews["has_image"].mean()]
})
display(summary)